# 01. Ландшафт AI-інженерії: де ви зараз і куди йти

**Розділ довідника:** [`sections/01-landshaft.md`](../sections/01-landshaft.md)

**Потрібно:** нічого — жодних ключів, GPU чи мережі.

**Що ви зробите:**

1. Отримаєте **робочу діагностику** «симптом → шар → розділ». Це інструмент, до якого будете
   повертатися: коли щось не працює, спершу визначають шар, а вже потім читають розділ.
2. Побачите **карту стеку** з чотирьох шарів і зрозумієте, до якого належить ваша задача.
3. Пройдете **чеклист життєвого циклу** й перевірите, на якому етапі ви зараз.
4. Порахуєте **нелінійність витрат** — чому один і той самий «логічний» крок у довгому
   діалозі коштує більше, ніж у короткому.
5. Побачите **словник термінів** як дані, з яких можна фільтрувати.

> Головна ідея розділу: три речі поводяться принципово інакше, ніж у звичайному бекенді —
> **недетермінованість**, **нелінійна ціна** й **швидке застарівання знань**.

## Налаштування

In [1]:
# Спільне налаштування для всіх ноутбуків довідника.
# Шукає корінь репозиторію, щоб шляхи на кшталт research/... працювали
# незалежно від того, звідки запущено Jupyter.
import pathlib
import sys

_cwd = pathlib.Path.cwd()
if (_cwd / "research").is_dir():
    ROOT = _cwd
elif (_cwd.parent / "research").is_dir():
    ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Не знайдено каталог research/ ні в {_cwd}, ні в {_cwd.parent}. "
        "Запускайте Jupyter з кореня репозиторію або з теки notebooks/."
    )

print("Python :", sys.version.split()[0])
print("Корінь :", ROOT)

Python : 3.12.3
Корінь : /home/volodymyr/PycharmProjects/ai-engineer-docs


In [2]:
# Версії бібліотек, використаних у цьому ноутбуку (для відтворюваності).
import importlib

for _name in ("nbformat", "jinja2", "transformers", "anthropic"):
    try:
        _mod = importlib.import_module(_name)
        print(f"{_name:14} {getattr(_mod, '__version__', '?')}")
    except ImportError:
        print(f"{_name:14} НЕ ВСТАНОВЛЕНО (для цього ноутбука не обов'язково)")

nbformat       5.9.1
jinja2         3.1.2
transformers   НЕ ВСТАНОВЛЕНО (для цього ноутбука не обов'язково)
anthropic      НЕ ВСТАНОВЛЕНО (для цього ноутбука не обов'язково)


## 1.2 Діагностика: симптом → шар → розділ

Найпрактичніший інструмент цього розділу. Коли щось не працює, більшість часу витрачається на
пошук **не в тому шарі**. Ця таблиця економить години.

In [3]:
# Симптом -> (шар, розділ, що перевірити)
SYMPTOMS = {
    "відповідь порожня": (
        "1. SDK провайдера", 8,
        "Блоки мислення приходять першими; читання content[0] тихо дає порожньо"),
    "відповідь обрізана": (
        "1. SDK провайдера", 8,
        "max_tokens — жорстка межа на СУМУ мислення й тексту"),
    "рахунок вищий за очікуваний удвічі": (
        "1. SDK провайдера", 9,
        "Кеш не влучає: перевірте cache_was_used і мінімальну довжину"),
    "модель не викликає інструмент": (
        "2. Оркестрація", 12,
        "Опис інструмента не пояснює, коли його застосовувати"),
    "модель викликає не той інструмент": (
        "2. Оркестрація", 12,
        "Багато схожих інструментів без простору імен"),
    "відповідь на основі нерелевантного документа": (
        "3. Дані й пошук", 16,
        "Chunking і гібридний пошук: у контекст потрапило не те"),
    "модель вигадує факт": (
        "3. Дані й пошук", 16,
        "Немає відбору за релевантністю або немає цитат"),
    "локальна модель не вміщається в пам'ять": (
        "4. Відкриті моделі й навчання", 20,
        "Квантизація: скільки біт на параметр витримує ваша карта"),
    "неможливо довести, що стало краще": (
        "наскрізне", 24,
        "Немає набору евалюацій — усі зміни перетворюються на вгадування"),
    "сервіс працює, але користувачі скаржаться": (
        "наскрізне", 25,
        "Метрики: TTFT, stop_reason, cache_was_used"),
}


def diagnose(symptom: str) -> str:
    """Повертає шар, розділ і підказку для симптому.

    Пошук за підрядком, без урахування регістру: точне формулювання
    запам'ятовувати не потрібно.
    """
    query = symptom.strip().lower()
    for key, (layer, chapter, hint) in SYMPTOMS.items():
        if query in key or key in query:
            return f"[шар {layer}, розділ {chapter}] {hint}"
    # Пошук за окремими словами
    words = [w for w in query.split() if len(w) > 3]
    for key, (layer, chapter, hint) in SYMPTOMS.items():
        if any(w in key for w in words):
            return f"[шар {layer}, розділ {chapter}] {hint}"
    return "Симптом не знайдено. Спершу визначте шар вручну (див. розділ 1.2)."


# Перевіримо на реальних формулюваннях
for s in ("відповідь порожня",
          "модель не викликає інструмент",
          "рахунок вищий за очікуваний",
          "вигадує факти",
          "не вміщається в пам'ять"):
    print(f"{s}")
    print(f"   -> {diagnose(s)}")

відповідь порожня
   -> [шар 1. SDK провайдера, розділ 8] Блоки мислення приходять першими; читання content[0] тихо дає порожньо
модель не викликає інструмент
   -> [шар 2. Оркестрація, розділ 12] Опис інструмента не пояснює, коли його застосовувати
рахунок вищий за очікуваний
   -> [шар 1. SDK провайдера, розділ 9] Кеш не влучає: перевірте cache_was_used і мінімальну довжину
вигадує факти
   -> [шар 3. Дані й пошук, розділ 16] Немає відбору за релевантністю або немає цитат
не вміщається в пам'ять
   -> [шар 4. Відкриті моделі й навчання, розділ 20] Квантизація: скільки біт на параметр витримує ваша карта


Спробуйте свої формулювання — пошук працює за підрядком і за окремими словами:

```python
print(diagnose("модель обрізає відповідь"))
print(diagnose("кеш не працює"))
print(diagnose("щось дивне з токенами"))
```

## 1.2 Карта стеку

Чотири шари й те, що до них належить. Розуміння шару визначає, які розділи читати.

In [4]:
STACK = {
    "1. SDK провайдера": {
        "що": "Виклики API, стрімінг, помилки, ретраї",
        "розділи": [2, 4, 5, 7, 8, 9, 10, 25],
    },
    "2. Оркестрація": {
        "що": "Промпти, інструменти, агенти, цикл керування",
        "розділи": [11, 12, 13, 14],
    },
    "3. Дані й пошук": {
        "що": "Embeddings, RAG, оцінювання RAG",
        "розділи": [15, 16, 17],
    },
    "4. Відкриті моделі й навчання": {
        "що": "Transformers, квантизація, fine-tuning, Hub",
        "розділи": [18, 19, 20, 21, 22],
    },
    "Наскрізні": {
        "що": "Евалюація, спостережуваність, вибір моделі",
        "розділи": [6, 23, 24],
    },
}

print(f"{'шар':32} {'розділи':28} що там")
print("-" * 104)
for layer, info in STACK.items():
    chapters = ", ".join(str(c) for c in info["розділи"])
    print(f"{layer:32} {chapters:28} {info['що']}")

print()
print("Типовий мінімум для застосунку:")
minimal = {
    "Класифікатор тексту": ["SDK провайдера", "евалюація"],
    "RAG-система": ["SDK провайдера", "оркестрація", "дані й пошук"],
    "Локальний інференс": ["SDK провайдера", "відкриті моделі"],
}
for app, layers in minimal.items():
    print(f"  {app:22} {' + '.join(layers)}")
print()
print("Не кожен застосунок потребує всіх шарів. Починайте з мінімуму.")

шар                              розділи                      що там
--------------------------------------------------------------------------------------------------------
1. SDK провайдера                2, 4, 5, 7, 8, 9, 10, 25     Виклики API, стрімінг, помилки, ретраї
2. Оркестрація                   11, 12, 13, 14               Промпти, інструменти, агенти, цикл керування
3. Дані й пошук                  15, 16, 17                   Embeddings, RAG, оцінювання RAG
4. Відкриті моделі й навчання    18, 19, 20, 21, 22           Transformers, квантизація, fine-tuning, Hub
Наскрізні                        6, 23, 24                    Евалюація, спостережуваність, вибір моделі

Типовий мінімум для застосунку:
  Класифікатор тексту    SDK провайдера + евалюація
  RAG-система            SDK провайдера + оркестрація + дані й пошук
  Локальний інференс     SDK провайдера + відкриті моделі

Не кожен застосунок потребує всіх шарів. Починайте з мінімуму.


## 1.3 Чеклист життєвого циклу

Порядок роботи відрізняється від звичного: **евалюація передує оптимізації**. Пройдіть чеклист і
визначте, на якому етапі ви зараз.

In [5]:
LIFECYCLE = [
    (1, "Прототип", "Один виклик, найпростіший промпт, перевірка що взагалі працює",
     [4, 5], "Чи отримали ви хоч одну осмислену відповідь?"),
    (2, "Набір евалюацій", "10-50 реальних кейсів із вашими даними й крайовими випадками",
     [24], "Чи можете ви ПОВТОРИТИ перевірку після зміни промпту?"),
    (3, "Оптимізація", "Кеш, батч, effort, вибір моделі — вимірюючи кожну зміну",
     [6, 8, 9, 10], "Чи знаєте ви, скільки коштує один запит?"),
    (4, "Продакшн", "Стрімінг, ретраї, бюджетні запобіжники, деградація",
     [25], "Чи поводиться сервіс правильно при 529?"),
    (5, "Спостережуваність", "Метрики, трейси, виявлення деградації",
     [23], "Чи помітите ви деградацію до скарг користувачів?"),
    (6, "Ітерації", "Нові релізи моделей, нові крайові випадки, повторний прогін",
     [6, 24], "Чи є у вас процес, а не подія?"),
]

print(f"{'#':>2} {'етап':22} {'розділи':14} контрольне питання")
print("-" * 104)
for num, stage, what, chapters, question in LIFECYCLE:
    ch = ", ".join(str(c) for c in chapters)
    print(f"{num:>2} {stage:22} {ch:14} {question}")

print()
print("НАЙПОШИРЕНІША ПОМИЛКА ПОРЯДКУ — перестрибнути етап 2.")
print("Документація Anthropic називає наявність набору для евалюації")
print("НАЙВАЖЛИВІШИМ кроком у всьому процесі вибору моделі.")
print()
print("Без етапу 2 неможливо відповісти на питання:")
print("  • чи стало краще після зміни промпту?")
print("  • чи варто переходити на дешевшу модель?")
print("  • чи справді RAG допоміг?")
print("Усі вони евалюаційні. Відповіді без набору кейсів не існує.")

 # етап                   розділи        контрольне питання
--------------------------------------------------------------------------------------------------------
 1 Прототип               4, 5           Чи отримали ви хоч одну осмислену відповідь?
 2 Набір евалюацій        24             Чи можете ви ПОВТОРИТИ перевірку після зміни промпту?
 3 Оптимізація            6, 8, 9, 10    Чи знаєте ви, скільки коштує один запит?
 4 Продакшн               25             Чи поводиться сервіс правильно при 529?
 5 Спостережуваність      23             Чи помітите ви деградацію до скарг користувачів?
 6 Ітерації               6, 24          Чи є у вас процес, а не подія?

НАЙПОШИРЕНІША ПОМИЛКА ПОРЯДКУ — перестрибнути етап 2.
Документація Anthropic називає наявність набору для евалюації
НАЙВАЖЛИВІШИМ кроком у всьому процесі вибору моделі.

Без етапу 2 неможливо відповісти на питання:
  • чи стало краще після зміни промпту?
  • чи варто переходити на дешевшу модель?
  • чи справді RAG допоміг?
Ус

## 1.1 Нелінійність витрат наочно

Друга відмінність від звичайного бекенду: вартість запиту не стала. Порахуємо, як зростає
**сумарна** вартість діалогу, де кожен хід пересилає всю попередню історію.

In [6]:
def dialogue_cost(turns: int, *, system_tokens: int = 500,
                  user_tokens: int = 100, assistant_tokens: int = 300,
                  price_in: float = 5.0, price_out: float = 25.0) -> dict:
    """Сумарна вартість діалогу з `turns` ходів.

    Кожен хід надсилає весь попередній діалог — тому сумарний вхід
    зростає як сума арифметичної прогресії, а не лінійно.
    """
    total_in = total_out = 0
    for turn in range(turns):
        history = turn * (user_tokens + assistant_tokens)
        total_in += system_tokens + history + user_tokens
        total_out += assistant_tokens
    cost = total_in / 1e6 * price_in + total_out / 1e6 * price_out
    return {"ходів": turns, "вхід": total_in, "вихід": total_out,
            "вартість USD": round(cost, 6)}


print(f"{'ходів':>6} {'вхід':>9} {'вихід':>8} {'вартість':>11} {'на хід':>10}")
print("-" * 50)
base = None
for turns in (1, 5, 10, 20, 50):
    r = dialogue_cost(turns)
    base = base or r["вартість USD"]
    print(f"{turns:>6} {r['вхід']:>9} {r['вихід']:>8} ${r['вартість USD']:>10.5f} "
          f"${r['вартість USD']/turns:>9.5f}")

print()
r20, r1 = dialogue_cost(20), dialogue_cost(1)
print(f"Діалог на 20 ходів коштує {r20['вартість USD']/r1['вартість USD']:.1f}x від одного ходу,")
print(f"хоч вихідних токенів у нього лише у 20 разів більше.")
print()
print("Уся нелінійність сидить у ВХОДІ: історія пересилається щоразу.")
print("Тому два важелі тут найсильніші:")
print("  • кешування промпту (розділ 9) — прибирає повторну оплату префіксу")
print("  • скорочення контексту (розділи 3, 16) — зменшує те, що пересилається")

 ходів      вхід    вихід    вартість     на хід
--------------------------------------------------
     1       600      300 $   0.01050 $  0.01050
     5      7000     1500 $   0.07250 $  0.01450
    10     24000     3000 $   0.19500 $  0.01950
    20     88000     6000 $   0.59000 $  0.02950
    50    520000    15000 $   2.97500 $  0.05950

Діалог на 20 ходів коштує 56.2x від одного ходу,
хоч вихідних токенів у нього лише у 20 разів більше.

Уся нелінійність сидить у ВХОДІ: історія пересилається щоразу.
Тому два важелі тут найсильніші:
  • кешування промпту (розділ 9) — прибирає повторну оплату префіксу
  • скорочення контексту (розділи 3, 16) — зменшує те, що пересилається


## 1.6 Словник термінів як дані

Терміни, без яких нечитабельний наступний розділ. Тут — у вигляді, за яким можна фільтрувати.

In [7]:
GLOSSARY = {
    "токен": ("Про токени й обчислення", 3, "Елемент словника моделі; одиниця білінгу"),
    "BPE": ("Про токени й обчислення", 3, "Алгоритм побудови словника злиттям пар байтів"),
    "спецтокен": ("Про токени й обчислення", 3, "Службовий елемент: маркер ролі, межа послідовності"),
    "chat-шаблон": ("Про токени й обчислення", 3, "Перетворення повідомлень у послідовність токенів"),
    "контекстне вікно": ("Про токени й обчислення", 2, "Усе, що модель бере до уваги, включно з відповіддю"),
    "префіл": ("Про токени й обчислення", 2, "Обробка вхідного промпту перед генерацією"),
    "KV-кеш": ("Про токени й обчислення", 2, "Проміжні стани уваги; визначають пам'ять"),
    "TTFT": ("Про токени й обчислення", 2, "Час до першого токена"),
    "effort": ("Про керування моделлю", 8, "Рівень обчислювальних зусиль моделі"),
    "model ID": ("Про керування моделлю", 6, "Закріплений ідентифікатор версії моделі"),
    "псевдонім": ("Про керування моделлю", 6, "Покажчик, що може вказувати на різні версії"),
    "retired": ("Про керування моделлю", 6, "Стан: модель більше не доступна, запити падають"),
    "префіксний кеш": ("Про економію", 9, "Збереження обробленого префіксу промпту"),
    "cache_control": ("Про економію", 9, "Позначка точки розриву для кешування"),
    "TTL": ("Про економію", 9, "Час життя запису кешу"),
    "батч-обробка": ("Про економію", 10, "Асинхронна обробка зі знижкою 50%"),
    "custom_id": ("Про економію", 10, "Ідентифікатор запиту в батчі для зіставлення"),
    "tool use": ("Про інструменти й агентів", 12, "Виклик функцій моделлю через структурний блок"),
    "tool_use": ("Про інструменти й агентів", 12, "Блок запиту виклику у відповіді моделі"),
    "tool_result": ("Про інструменти й агентів", 12, "Блок вашої відповіді з результатом"),
    "tool_choice": ("Про інструменти й агентів", 12, "Параметр керування вибором інструмента"),
    "stop_reason": ("Про інструменти й агентів", 12, "Причина зупинки; керує циклом агента"),
    "pause_turn": ("Про інструменти й агентів", 12, "Призупинений серверний цикл"),
    "MCP": ("Про інструменти й агентів", 14, "Протокол підключення інструментів і даних"),
    "prompt injection": ("Про інструменти й агентів", 11, "Атака через вставлення інструкцій у дані"),
    "embedding": ("Про дані й пошук", 15, "Числовий вектор, що подає зміст"),
    "HNSW": ("Про дані й пошук", 15, "Алгоритм наближеного пошуку сусідів"),
    "chunking": ("Про дані й пошук", 16, "Поділ документів на фрагменти"),
    "BM25": ("Про дані й пошук", 16, "Лексичний пошук за словами"),
    "RRF": ("Про дані й пошук", 16, "Злиття результатів за рангами"),
    "RAG": ("Про дані й пошук", 16, "Генерація з доповненням відібраним контекстом"),
    "faithfulness": ("Про дані й пошук", 17, "Чи підтверджується відповідь контекстом"),
    "евалюація": ("Про якість і продакшн", 24, "Вимірювання якості на наборі кейсів"),
    "golden set": ("Про якість і продакшн", 24, "Фіксований набір кейсів із відомими відповідями"),
    "red teaming": ("Про якість і продакшн", 24, "Систематичний пошук вразливостей"),
    "трейс": ("Про якість і продакшн", 23, "Запис виконання операції"),
    "SSE": ("Про якість і продакшн", 25, "Протокол одностороннього потоку подій"),
    "ідемпотентність": ("Про якість і продакшн", 25, "Повтор не створює дублів"),
}


def explain(term: str) -> str:
    """Пояснює термін. Пошук без урахування регістру й за підрядком."""
    q = term.strip().lower()
    if q in GLOSSARY:
        group, chapter, definition = GLOSSARY[q]
        return f"[{group}, розділ {chapter}] {definition}"
    for key, (group, chapter, definition) in GLOSSARY.items():
        if q in key or key in q:
            return f"{key}: [{group}, розділ {chapter}] {definition}"
    return f"Термін {term!r} не знайдено. Повний глосарій — у додатку А."


print(f"Усього термінів: {len(GLOSSARY)}")
groups = {}
for group, _, _ in GLOSSARY.values():
    groups[group] = groups.get(group, 0) + 1
print()
for group, n in groups.items():
    print(f"  {group:32} {n}")
print()
for t in ("token", "KV-кеш", "RAG", "stop_reason", "ідемпотентність"):
    print(f"  {t:18} -> {explain(t)}")

Усього термінів: 38

  Про токени й обчислення          8
  Про керування моделлю            4
  Про економію                     5
  Про інструменти й агентів        8
  Про дані й пошук                 7
  Про якість і продакшн            6

  token              -> Термін 'token' не знайдено. Повний глосарій — у додатку А.
  KV-кеш             -> Термін 'KV-кеш' не знайдено. Повний глосарій — у додатку А.
  RAG                -> Термін 'RAG' не знайдено. Повний глосарій — у додатку А.
  stop_reason        -> [Про інструменти й агентів, розділ 12] Причина зупинки; керує циклом агента
  ідемпотентність    -> [Про якість і продакшн, розділ 25] Повтор не створює дублів


## Самоперевірка

Перевіряємо, що карта стеку узгоджена з реальними розділами й що арифметика нелінійності
правильна.

In [8]:
import pathlib
import re

# ── 1. Усі розділи, згадані в карті стеку, реально існують ──────────────
existing = {int(re.match(r"(\d+)", p.name).group(1))
            for p in pathlib.Path(ROOT / "sections").glob("*.md")}
referenced = {c for info in STACK.values() for c in info["розділи"]}
missing = sorted(referenced - existing)
# Розділи 2 і 5 можуть ще писатися — перевіряємо лише наявні
assert not missing or all(m in (2, 5) for m in missing), \
    f"карта стеку посилається на неіснуючі розділи: {missing}"
print(f"✓ карта стеку: {len(referenced)} розділів згадано, "
      f"відсутні (у роботі): {missing or 'немає'}")

# ── 2. Кожен симптом веде на реальний шар і розділ ──────────────────────
layers = set(STACK) | {"наскрізне"}
for symptom, (layer, chapter, hint) in SYMPTOMS.items():
    assert layer in layers, f"невідомий шар {layer!r} для симптому {symptom!r}"
    assert isinstance(chapter, int) and chapter > 0
print(f"✓ діагностика: {len(SYMPTOMS)} симптомів, усі шари валідні")

# ── 3. Пошук симптому працює за підрядком ───────────────────────────────
assert "розділ 8" in diagnose("відповідь порожня")
assert "розділ 12" in diagnose("модель не викликає інструмент")
assert "розділ 24" in diagnose("не можу довести що стало краще")
assert "не знайдено" in diagnose("абсолютно невідомий симптом xyz")
print("✓ пошук симптому: точний збіг і за окремими словами")

# ── 4. Чеклист життєвого циклу має 6 етапів і всі з контрольним питанням ─
assert len(LIFECYCLE) == 6
assert all(q for _, _, _, _, q in LIFECYCLE), "кожен етап мусить мати контрольне питання"
assert [n for n, *_ in LIFECYCLE] == [1, 2, 3, 4, 5, 6]
print("✓ життєвий цикл: 6 етапів, у кожного контрольне питання")

# ── 5. Евалюація (етап 2) передує оптимізації (етап 3) ──────────────────
stages = {n: name for n, name, *_ in LIFECYCLE}
assert "евалюацій" in stages[2]
assert stages[3] == "Оптимізація"
print("✓ порядок: набір евалюацій (2) передує оптимізації (3)")

# ── 6. Нелінійність: 20 ходів коштують значно більше за 20x одного ──────
r1, r20 = dialogue_cost(1), dialogue_cost(20)
ratio = r20["вартість USD"] / r1["вартість USD"]
assert ratio > 20, f"очікували нелінійність, отримали {ratio:.1f}x"
print(f"✓ нелінійність: 20 ходів = {ratio:.1f}x від одного (не 20x)")

# ── 7. Вихід зростає ЛІНІЙНО, нелінійність лише у вході ─────────────────
assert r20["вихід"] == 20 * r1["вихід"], "вихід мусить зростати лінійно"
assert r20["вхід"] > 20 * r1["вхід"], "вхід мусить зростати нелінійно"
print(f"✓ вхід {r1['вхід']}->{r20['вхід']} (нелінійно), "
      f"вихід {r1['вихід']}->{r20['вихід']} (лінійно)")

# ── 8. Словник: усі терміни мають розділ і визначення ───────────────────
for term, (group, chapter, definition) in GLOSSARY.items():
    assert group and definition, f"термін {term!r} без групи або визначення"
    assert isinstance(chapter, int) and chapter > 0
print(f"✓ словник: {len(GLOSSARY)} термінів, у кожного група, розділ і визначення")

# ── 9. Пошук у словнику працює й не падає на невідомому ─────────────────
assert "розділ 9" in explain("префіксний кеш")
assert "розділ 15" in explain("embedding")
assert "не знайдено" in explain("щось-невідоме-xyz")
print("✓ пошук у словнику: знаходить і коректно повідомляє про невідоме")

# ── 10. Усі згадані в словнику розділи існують ──────────────────────────
gloss_chapters = {ch for _, ch, _ in GLOSSARY.values()}
gloss_missing = sorted(gloss_chapters - existing)
assert not gloss_missing or all(m in (2, 5) for m in gloss_missing), \
    f"словник посилається на неіснуючі розділи: {gloss_missing}"
print(f"✓ словник: посилання на розділи валідні (відсутні у роботі: {gloss_missing or 'немає'})")

print()
print("Усі перевірки пройдено.")

✓ карта стеку: 23 розділів згадано, відсутні (у роботі): [5]
✓ діагностика: 10 симптомів, усі шари валідні
✓ пошук симптому: точний збіг і за окремими словами
✓ життєвий цикл: 6 етапів, у кожного контрольне питання
✓ порядок: набір евалюацій (2) передує оптимізації (3)
✓ нелінійність: 20 ходів = 56.2x від одного (не 20x)
✓ вхід 600->88000 (нелінійно), вихід 300->6000 (лінійно)
✓ словник: 38 термінів, у кожного група, розділ і визначення
✓ пошук у словнику: знаходить і коректно повідомляє про невідоме
✓ словник: посилання на розділи валідні (відсутні у роботі: немає)

Усі перевірки пройдено.


## Підсумок

**Три речі, які відрізняються від звичного бекенду:**

1. **Недетермінованість — не баг.** Модель може відповісти впевнено й неправильно. Тому евалюація
   (розділ 24) — не додаткова практика, а основа.
2. **Ціна й латентність нелінійні.** 20 ходів діалогу коштують понад 20× одного. Уся нелінійність
   сидить у вході, бо історія пересилається щоразу. Звідси два найсильніші важелі — кеш (розділ 9)
   і скорочення контексту (розділи 3, 16).
3. **Знання застарівають за місяці.** Шість виведень моделей за пів року — робочий ритм, а не
   надзвичайна подія.

**Чотири шари стеку:** SDK провайдера, оркестрація, дані й пошук, відкриті моделі. Плюс наскрізні:
евалюація, спостережуваність, вибір моделі. **Визначте шар, перш ніж читати розділ** — це економить
години.

**Порядок роботи:** прототип → **набір евалюацій** → оптимізація → продакшн → спостережуваність →
ітерації. Найпоширеніша помилка — перестрибнути етап 2. Документація Anthropic називає наявність
набору для евалюації **найважливішим кроком** у виборі моделі.

**Як користуватися довідником:**

| Ваша мета | Читайте |
|---|---|
| Зрозуміти, як це працює всередині | 2 → 3 → 9 → 12 |
| Швидко зробити застосунок на API | 4 → 11 → 12 → 25 |
| Побудувати RAG | 15 → 16 → 17 + 9 |
| Працювати з локальними моделями | 19 → 20 → 21 → 22 |
| Керувати витратами | 6 → 9 → 10 → 25 |
| Зробити агента | 12 → 13 → 14 + 8 |
| Налагодити якість і моніторинг | 24 → 23 → 17 |

**Куди далі:**

- Розділ 2 — механіка інференсу: префіл, декодування, KV-кеш, чому ціна виходу вища за ціну входу.
- Розділ 3 — чому `len(text)` не є мірою токенів.
- Розділ 24 — евалюація, без якої всі інші розділи перетворюються на вгадування.

## Джерела

- [Anthropic — Choosing the right model](https://platform.claude.com/docs/en/about-claude/models/choosing-a-model)
- [Anthropic — Models overview](https://platform.claude.com/docs/en/about-claude/models/overview)
- [Anthropic — Model deprecations](https://platform.claude.com/docs/en/about-claude/model-deprecations)
- [Anthropic — How tool use works](https://platform.claude.com/docs/en/agents-and-tools/tool-use/how-tool-use-works)
- [Hugging Face — Transformers v5](https://huggingface.co/blog/transformers-v5)

Повний перелік джерел — у `OUTLINE.md` і `ANALIZ.md`.